#### Importing libraries

In [ ]:
import cv2
import numpy as np
import os
from PIL import Image
import glob
import re


#### Utility functions

In [ ]:
def natural_sort_key(filename):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', filename)]

def extract_text_lines_from_image(image_path, text_path, line_counter, anno_file, output_folder='line_images'):
    """
    Extract line-level images from a single page image and map them to text lines.
    """
    # Read the image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Error: Could not load image from {image_path}")
        return line_counter
    
    # --- Thresholding (Otsu + adjustment) ---
    if len(np.unique(img)) > 2:
        otsu_threshold, _ = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        mean_intensity = np.mean(img)
        if mean_intensity > 200:
            threshold_value = min(otsu_threshold + 20, 200)
        elif mean_intensity > 150:
            threshold_value = min(otsu_threshold + 10, 180)
        else:
            threshold_value = max(otsu_threshold - 10, 100)
        _, img = cv2.threshold(img, threshold_value, 255, cv2.THRESH_BINARY)
        print(f"Applied binarization with threshold: {threshold_value} (Otsu: {otsu_threshold}, Mean: {mean_intensity:.1f})")

    # --- Ensure black text on white background ---
    white_pixels = np.sum(img == 255)
    black_pixels = np.sum(img == 0)
    if black_pixels > white_pixels:
        img = cv2.bitwise_not(img)

    # --- Line segmentation via horizontal projection ---
    binary_img = (img == 0).astype(np.uint8)  # text = 1, background = 0
    horizontal_projection = np.sum(binary_img, axis=1)

    line_boundaries = []
    in_line = False
    start_row = 0

    max_projection = np.max(horizontal_projection)
    mean_projection = np.mean(horizontal_projection[horizontal_projection > 0]) if np.any(horizontal_projection > 0) else 0
    threshold = max(max_projection * 0.2, mean_projection * 0.5) if max_projection > 0 else 0

    for i, projection in enumerate(horizontal_projection):
        if projection > threshold and not in_line:
            start_row = i
            in_line = True
        elif projection <= threshold and in_line:
            line_boundaries.append((start_row, i))
            in_line = False
    if in_line:
        line_boundaries.append((start_row, len(horizontal_projection)))

    # --- Read corresponding text file ---
    text_lines = []
    if os.path.exists(text_path):
        with open(text_path, "r", encoding="utf-8") as f:
            text_lines = [line.strip() for line in f.readlines()]
    else:
        print(f"⚠ No matching text file for {os.path.basename(image_path)}")

    # --- Extract and save each line ---
    lines_extracted = 0
    used_text_indices = set()

    for idx, (start, end) in enumerate(line_boundaries):
        padding = 5
        start_padded = max(0, start - padding)
        end_padded = min(img.shape[0], end + padding)

        line_img = img[start_padded:end_padded, :]

        # Trim side whitespace
        vertical_projection = np.sum((line_img == 0).astype(np.uint8), axis=0)
        non_empty_cols = np.where(vertical_projection > 0)[0]
        if len(non_empty_cols) == 0:
            continue
        left, right = non_empty_cols[0], non_empty_cols[-1] + 1
        line_img_cropped = line_img[:, left:right]

        if line_img_cropped.shape[0] < 10 or line_img_cropped.shape[1] < 20:
            continue

        # Save line image as JPG
        line_img_name = f"{line_counter}_nataksamaysarbhasha_101_137.jpg"
        output_path = os.path.join(output_folder, line_img_name)
        Image.fromarray(line_img_cropped).save(output_path, format="JPEG", quality=95)

        # Map to text line if available
        if idx < len(text_lines):
            anno_file.write(f"{line_img_name}\t{text_lines[idx]}\n")
            used_text_indices.add(idx)
        else:
            anno_file.write(f"{line_img_name}\t[NO_TEXT]\n")

        print(f"Saved line {line_counter}: {output_path}")
        line_counter += 1
        lines_extracted += 1

    # --- Write unmapped text lines as NULL images ---
    for i, txt in enumerate(text_lines):
        if i not in used_text_indices:
            anno_file.write(f"[NO_IMAGE]\t{txt}\n")

    print(f"Extracted {lines_extracted} lines from {os.path.basename(image_path)}")
    return line_counter

def process_folder(image_folder, text_folder, output_folder='line_images', anno_filename='annotations.txt'):
    """
    Process all images in folder and map extracted lines to text.
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Collect image files
    exts = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tiff', '*.tif']
    image_files = []
    for ext in exts:
        image_files.extend(glob.glob(os.path.join(image_folder, ext)))
        image_files.extend(glob.glob(os.path.join(image_folder, ext.upper())))
    if not image_files:
        print(f"No images found in {image_folder}")
        return

    image_files.sort(key=lambda x: natural_sort_key(os.path.basename(x)))
    print(f"Found {len(image_files)} images to process")

    anno_path = os.path.join(output_folder, anno_filename)
    with open(anno_path, "w", encoding="utf-8") as anno_file:
        line_counter = 1
        for i, image_path in enumerate(image_files, 1):
            page_name = os.path.splitext(os.path.basename(image_path))[0]
            text_path = os.path.join(text_folder, f"{page_name}.txt")
            print(f"\nProcessing {i}/{len(image_files)}: {os.path.basename(image_path)}")
            line_counter = extract_text_lines_from_image(image_path, text_path, line_counter, anno_file, output_folder)

    print(f"\n✅ Processing complete! Annotations saved to {anno_path}")



#### Main function

In [ ]:
if __name__ == "__main__":
    image_folder = "../manuscript_dataset/INCORRECT/nataksamaysarbhasha_101_137/page_level/images"
    text_folder = "../manuscript_dataset/INCORRECT/nataksamaysarbhasha_101_137/line_level/text"
    output_folder = "../manuscript_dataset/INCORRECT/nataksamaysarbhasha_101_137/line_level/line_images"
    process_folder(image_folder, text_folder, output_folder)
